In [1]:
import clickhouse_connect

from clickhouse_connect.driver.tools import insert_file
import pyarrow
import pandas as pd


In [2]:
client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=1300)

client.command("""
Drop Table idnet
""")

In [11]:
client.command("""
create table IF NOT EXISTS idnet100k_no_dupe
(
    name             text,
    imageData      text,
    prelabeled_fraud integer,
    id               text,
    person_name      text,
    address          text,
    birthday         text,
    gender           text,
    ethnicity        Nullable(text),
    class            text,
    issue_date       text,
    expire_date      text,
    eye_color        text,
    hair_color       Nullable(text),
    is_donor         text,
    is_veteran       text,
    license_number   text,
    height_inches    integer,
    weight_lbs       integer
)
    ENGINE = MergeTree
ORDER BY name
SETTINGS allow_nullable_key = 1
""")

In [12]:
#client.command()

In [3]:
df = pd.read_parquet('../silver/idnet100k_no_dupe.parquet')

In [4]:
print(df.columns)

Index(['address', 'birthday', 'class', 'ethnicity', 'expire_date', 'eye_color',
       'gender', 'hair_color', 'height_inches', 'id', 'imageData', 'is_donor',
       'is_veteran', 'issue_date', 'license_number', 'name', 'person_name',
       'prelabeled_fraud', 'weight_lbs'],
      dtype='object')


In [14]:
table_name = "idnet100k_no_dupe"
chunk_size = 1000  # Adjust based on memory and row width

In [12]:
client.command("truncate table idnet100k_no_dupe SETTINGS max_table_size_to_drop = 0")

In [16]:
client.command("select count(*) from idnet100k_no_dupe")

106060

In [21]:
client.close()

In [15]:
import time

total_rows = len(df)
print(f"Starting batch insert of {total_rows:,} rows...")

for start_idx in range(0, total_rows, chunk_size):
    # Slice the dataframe into a smaller chunk
    end_idx = min(start_idx + chunk_size, total_rows)
    df_chunk = df.iloc[start_idx:end_idx]

    #client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=1300)

    # Insert the chunk
    client.insert_df(table=table_name, df=df_chunk, settings={
        "send_timeout": 1300,            # Max time server spends sending data
        "receive_timeout": 1300,         # Max time server spends receiving data from client
        "max_execution_time": 1300,      # Prevents ClickHouse from aborting long-running queries
        "timeout_before_checking_execution_speed": 1300,  # Stops aborts from low-speed thresholds
        "connect_timeout": 1300, })

    #client.close()

    time.sleep(60)
    print(f"Successfully inserted rows {start_idx:,} to {end_idx:,}")

Starting batch insert of 106,060 rows...
Successfully inserted rows 0 to 1,000
Successfully inserted rows 1,000 to 2,000
Successfully inserted rows 2,000 to 3,000
Successfully inserted rows 3,000 to 4,000
Successfully inserted rows 4,000 to 5,000
Successfully inserted rows 5,000 to 6,000
Successfully inserted rows 6,000 to 7,000
Successfully inserted rows 7,000 to 8,000
Successfully inserted rows 8,000 to 9,000
Successfully inserted rows 9,000 to 10,000
Successfully inserted rows 10,000 to 11,000
Successfully inserted rows 11,000 to 12,000
Successfully inserted rows 12,000 to 13,000
Successfully inserted rows 13,000 to 14,000
Successfully inserted rows 14,000 to 15,000
Successfully inserted rows 15,000 to 16,000
Successfully inserted rows 16,000 to 17,000
Successfully inserted rows 17,000 to 18,000
Successfully inserted rows 18,000 to 19,000
Successfully inserted rows 19,000 to 20,000
Successfully inserted rows 20,000 to 21,000
Successfully inserted rows 21,000 to 22,000
Successfully i

In [8]:
client.close()

In [ ]:
client.command('select * from idnet10k_no_dupe limit 1')

In [ ]:
#stop here for data load

In [25]:
client.command('SYSTEM RELOAD FUNCTIONS')

In [27]:
client.command('select test_function_python(2.0)')

'Value 2'

In [13]:
client.command('select test_function_python_venv(2.0)')

DatabaseError: Received ClickHouse exception, code: 46, server response: Code: 46. DB::Exception: Function with name `test_function_python_venv` does not exist. In scope SELECT test_function_python_venv(2.). Maybe you meant: ['test_function_python','test2_function_python']. (UNKNOWN_FUNCTION) (version 25.8.4.13 (official build)) (for url http://localhost:8123)

In [5]:
client.command("select torch_function_python(name) from idnet limit 1;")

DatabaseError: Received ClickHouse exception, code: 754, server response: Code: 754. DB::Exception: User defined function 'torch_function_python' failed. Command: 'clickhousetorch.py', Arguments: []. Original error: Code: 302. DB::Exception: Child process was exited with return code 1. (CHILD_WAS_NOT_EXITED_NORMALLY),. (UDF_EXECUTION_FAILED) (version 25.8.4.13 (official build)) (for url http://localhost:8123)

In [7]:
client.command('select * from idnet limit 1')

['generated.photos_v3_0000170',
 '...\\ncY/jMT4DwP8LiTbVZ1W9ua4AAAAASUVORK5CYII=',
 '0',
 'generated.photos_v3_0000170',
 'BRIAN WADE',
 '1436 Deer Valley Road, Peoria, AZ 85345',
 '1991-06-05',
 'M',
 'white',
 'D',
 '2022-10-09',
 '2027-10-09',
 'BRO',
 'BRO',
 'FALSE',
 'FALSE',
 'D82989586',
 '69',
 '118']

In [28]:
result = client.query('select name, xgboost_function_python(toString(gender), toString(ethnicity), toString(class), toString(eye_color), toString(hair_color), toString(is_donor), toString(is_veteran), toString(height_inches), toString(weight_lbs)) from idnet').result_rows
df = pd.DataFrame(result)
df

DatabaseError: Received ClickHouse exception, code: 349, server response: Code: 349. DB::Exception: Cannot convert NULL value to non-Nullable type: while executing 'FUNCTION xgboost_function_python(toString(__table1.gender) :: 10, toString(__table1.ethnicity) :: 0, toString(__table1.class) :: 1, toString(__table1.eye_color) :: 2, toString(__table1.hair_color) :: 3, toString(__table1.is_donor) :: 4, toString(__table1.is_veteran) :: 5, toString(__table1.height_inches) :: 6, toString(__table1.weight_lbs) :: 7) -> xgboost_function_python(toString(__table1.gender), toString(__table1.ethnicity), toString(__table1.c... Float32 : 8'. (CANNOT_INSERT_NULL_IN_ORDINARY_COLUMN) (version 25.8.4.13 (official build)) (for url http://localhost:8123)

In [ ]:
client.query('select test2_function_python(toString(gender), toString(ethnicity), toString(class), toString(eye_color), toString(hair_color), toString(is_donor), toString(is_veteran), toString(height_inches), toString(weight_lbs)) from idnet limit 5')


In [ ]:
client.query("select xgboost_function_python('M', 'white', 'D', 'BRO', 'BRO', 'FALSE', 'FALSE', '60', '120')")

In [ ]:
results = client.query("""
                       with cte as (
                           select name, torch_function_python(toString(name)) as pred from idnet
                           )
                       select * from cte where pred > 0.5
                       """).result_rows

In [ ]:
client.command('select * from idnet limit 1')

In [ ]:
for x in results:
    print(x)

In [ ]:
result = client.query("select gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran , height_inches, weight_lbs from idnet").result_rows
df = pd.DataFrame(result)
df

In [ ]:
result = client.query("select test2_function_python('M', 'white', 'D', 'BRO', 'BRO', 'FALSE', 'FALSE', '60', '120')").result_rows

print(result)

In [ ]:
result = client.query("select test2_function_python(gender, ethnicity, class, eye_color, hair_color, IF(is_donor == 'TRUE', 1,0) as is_donor, IF(is_veteran == 'TRUE', 1,0) as is_veteran, height_inches, weight_lbs) from idnet").result_rows
df = pd.DataFrame(result)
df


In [ ]:
result = client.query('select name, test2_function_python(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from idnet').result_rows
df = pd.DataFrame(result)
df